<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="40%"></a>
</p>

<span style="float:right">
<strong>Credits:</strong>
This notebook adapts material from the <a href="https://docs.ros.org/en/jazzy/">ROS 2 documentation</a> (<a href="https://creativecommons.org/licenses/by/4.0/">CC BY 4.0</a>) and from the Duckietown <em>ROS Basics</em> and <em>ROS 2 Middleware</em> learning experiences.
</span>

# ROS 2 Publishers: from a Timer to a Driving Duckiebot

In notebook 07, the local graph inspection showed the node `/my_publisher_node` publishing text on `/chatter` once per second. Where does the node acquire its name, what triggers each publication, and how does a source change become visible to a subscriber?

This notebook first follows `my_publisher_node` from initialisation through callback execution and shutdown. It then applies the same structure to the node you will use on the robot, `drive_node`, which publishes car commands on `car_cmd_switch_node/cmd`: you will read it section by section, start it, and drive the Duckiebot from this notebook with sliders and buttons.

## Part 1: anatomy of a publisher

### Find the behaviour in the source

Open [my_publisher_node.py](../packages/ros2_basics/ros2_basics/my_publisher_node.py) in the editor (right-click the file tab and choose *Split Right* to see the notebook and the source side by side). Near the beginning, the module defines:

```python
NODE_NAME = "my_publisher_node"
TOPIC_NAME = "chatter"
QUEUE_DEPTH = 10
```

These settings should now look familiar: the graph exposed `/my_publisher_node`, its topic resolved to `/chatter`, and the verbose topic info showed `KEEP_LAST (10)`.

The `MyPublisherNode` class inherits ROS 2 functionality from `rclpy.node.Node`. Its constructor begins by assigning the node name:

```python
super().__init__(NODE_NAME)
```

It then declares its two parameters with the `LiveParams` helper of this learning experience, which declares them, keeps their current value in `self.params.<name>` and calls `_on_param` whenever somebody changes one (that is what reacted to `ros2 param set` in notebook 07):

```python
self.params = LiveParams(self, {"message": "Hello from ROS 2!", "rate_hz": 1.0}, on_change=self._on_param)
```

Then it creates the publishing endpoint:

```python
self._publisher = self.create_publisher(String, TOPIC_NAME, QUEUE_DEPTH)
```

The arguments connect the message type, topic name and history depth to the publisher inspected earlier.

### What triggers a publication?

Creating the publisher prepares an endpoint, but repeated publication requires something to invoke the code that sends each message. The constructor establishes that schedule:

```python
self._timer = self.create_timer(1.0 / self.params.rate_hz, self._publish_message)
```

The timer period is expressed in seconds: at `1.0` Hz the period is one second; at `2.0` Hz, half a second. `self._publish_message` is passed as a **callback**, a function that ROS 2 invokes when work is ready. It appears *without parentheses*, because the constructor supplies the function to the timer for later execution rather than calling it now.

When called, the method creates and fills a message, logs it and publishes it:

```python
msg = String()
msg.data = f"{self.params.message} [{self._count}]"
self._count += 1
self.get_logger().info(f'publishing: "{msg.data}"')
self._publisher.publish(msg)
```

The log appears before the call to `publish`, which is why receiving the message through `echo` provides evidence beyond the publisher's own terminal output.

### Understand the node lifecycle

Lifecycle here refers to the execution of this ordinary node from initialisation through shutdown. The package entry point calls `main`:

```python
rclpy.init(args=args)
node = MyPublisherNode()
try:
    rclpy.spin(node)
except KeyboardInterrupt:
    pass
finally:
    node.destroy_node()
    if rclpy.ok():
        rclpy.shutdown()
```

`rclpy.init` prepares the ROS 2 client library, and constructing the node creates the publisher and the timer. During `rclpy.spin(node)`, an **executor** waits for ready work and invokes callbacks, including the timer callback and the parameter callback. The repeated publications occur while spinning continues, as illustrated in Figure 1.

<figure style="margin:1.5em auto; text-align:center;">
  <pre style="display:inline-block; margin:0; text-align:left;">
    rclpy.init()
      |
      v
    construct the node
      declare parameters, create publisher and timer
      |
      v
    rclpy.spin(node)
      wait for ready work
      invoke callbacks repeatedly (timer, parameter changes, subscriptions, services)
      |
      v
    interruption (Ctrl-C) or exit
      |
      v
    destroy the node and shut down the context
  </pre>
  <figcaption style="font-size:0.9em; margin-top:0.6em;">
    Figure 1: Execution of a node.
  </figcaption>
</figure>

Destroying the node releases its publisher and timer. The final condition checks whether the ROS 2 context still needs shutdown, since interrupt handling may already have shut it down.

Callbacks run one after the other in the executor: a callback that takes a long time (decoding an image, sleeping) delays every other callback of the node, including the timer. Keep callbacks short.

### Change the default text and rebuild

In notebook 07 the text was changed at run time with `ros2 param set`. A change that should survive a restart belongs in the source. Two places define the default: the Python dictionary given to `LiveParams`, and the yaml file used by the launch file. Change the yaml first. Open [config/my_publisher_node/default.yaml](../packages/ros2_basics/config/my_publisher_node/default.yaml) and set:

```yaml
message: "Hello from the publisher!"
rate_hz: 2.0
```

Save the file, then rebuild from Terminal 1 (`LX_DIR=/code/src/lx-ros2-basics` as in notebook 06):

```bash
cd "$COLCON_WS"
colcon build --base-paths "$LX_DIR/packages" --packages-select ros2_basics
source install/setup.bash
```

A yaml file is only applied when the node is started through its launch file, so this time start the publisher with `ros2 launch` instead of `ros2 run`:

```bash
ros2 launch ros2_basics my_publisher_node.launch.py veh:=lab
```

`veh:=lab` gives a value to the `veh` launch argument: the node now runs in the namespace `/lab`. The log should contain `Hello from the publisher!` twice per second, under the name `my_publisher_node`. In Terminal 2:

```bash
ros2 topic list -t --no-daemon --spin-time 5
ros2 topic echo --once --timeout 10 /lab/chatter std_msgs/msg/String
```

The topic became `/lab/chatter` (relative name `chatter` in namespace `/lab`), and the changed text arrives:

```shell
data: Hello from the publisher! [7]
---
```

This connects the source edit, the rebuilt installation, the restarted program and the received message. If the original text still appears, check that the file was saved, that the build succeeded, that the old publisher was stopped, and that the node was started through the launch file. Stop the publisher with `Ctrl-C`.

## Part 2: a node that drives the Duckiebot

The old joystick demo of Duckietown mapped joystick axes to wheel speeds. In this learning experience the Duckiebot is driven by a node whose behaviour is controlled through the two other interfaces of notebook 02: **parameters** (`v`, `omega`) and **services** (`start`, `stop`). Open [drive_node.py](../packages/ros2_basics/ros2_basics/drive_node.py) and read it top to bottom with the help of this section.

### Imports

```python
import rclpy
from duckietown_msgs.msg import Twist2DStamped
from rclpy.node import Node
from std_srvs.srv import Trigger

from lx_ros2_tools.params import LiveParams
from lx_ros2_tools.robot import get_wheel_params
```

`Twist2DStamped` is the message inspected in notebook 03 (`v` in m/s, `omega` in rad/s). `Trigger` is a standard service type with an empty request and a response containing `success` and `message`; it is the simplest way to tell a node "do it now". `get_wheel_params` reads the wheel baseline and radius from the robot's calibration file (with built-in defaults): not needed to drive, but it shows where such numbers come from.

### The constructor

```python
super().__init__("drive_node")
self.veh = self.get_namespace().strip("/")
```

The node does not read the robot name from a variable: it asks ROS 2 for its own namespace, which the launch file sets to the robot name. The same code works on every robot.

```python
self.params = LiveParams(self, {"v": 0.2, "omega": 0.0, "rate_hz": 10.0, "max_duration": 5.0, "enabled": False},
                         on_change=self._on_param)
```

Five parameters: the command to send, the publication rate, a safety limit (`max_duration` seconds after `start` the node stops by itself; `0` disables the limit) and the `enabled` flag that the services toggle.

```python
self.pub_cmd = self.create_publisher(Twist2DStamped, "car_cmd_switch_node/cmd", 10)
```

The publisher uses the *relative* name `car_cmd_switch_node/cmd`. In the robot's namespace it resolves to `/ROBOTNAME/car_cmd_switch_node/cmd`, the topic `kinematics_node` reads (notebook 08).

```python
self.create_service(Trigger, "~/start", self.srv_start)
self.create_service(Trigger, "~/stop", self.srv_stop)
```

Two services with *private* names: `/ROBOTNAME/drive_node/start` and `/ROBOTNAME/drive_node/stop`. Each service has a callback that receives the request and fills the response.

```python
self._timer = self.create_timer(1.0 / self.params.rate_hz, self.loop)
```

The control loop is a timer callback, exactly like the text publisher.

### The control loop

```python
def loop(self) -> None:
    if not self.params.enabled:
        return
    now = self.get_clock().now()
    elapsed = (now - self._started_at).nanoseconds * 1e-9 ...
    if self.params.max_duration > 0 and elapsed > self.params.max_duration:
        self.stop()
        return

    cmd = Twist2DStamped()
    cmd.header.stamp = now.to_msg()
    # YOUR CODE BELOW THIS LINE
    cmd.v = float(self.params.v)
    cmd.omega = float(self.params.omega)
    # YOUR CODE ABOVE THIS LINE
    self.pub_cmd.publish(cmd)
```

Ten times per second, if driving is enabled and the time limit is not reached, the loop builds a command with the current time stamp and the current parameter values and publishes it. The block between the markers is the only place you need to touch to change *what* the robot does; the exercise at the end of this notebook does that.

### The services

```python
def srv_start(self, _request, response):
    self._started_at = self.get_clock().now()
    self.params.enabled = True
    response.success = True
    response.message = f"driving with v={self.params.v} m/s, omega={self.params.omega} rad/s"
    return response
```

`start` records the time and enables the loop. Assigning `self.params.enabled` goes through the ROS 2 parameter API, so `ros2 param get` and the panels see the change. `stop` disables the loop and publishes one zero command, so the wheels stop immediately rather than at the next time-out of the motor driver.

### The launch file and the configuration

[launch/drive_node.launch.py](../packages/ros2_basics/launch/drive_node.launch.py) starts the node in the robot's namespace with the defaults of [config/drive_node/default.yaml](../packages/ros2_basics/config/drive_node/default.yaml):

```python
DeclareLaunchArgument("veh", default_value=EnvironmentVariable("VEHICLE_NAME"), ...),
Node(package="ros2_basics", executable="drive_node", name="drive_node",
     namespace=veh, parameters=[config],
     remappings=[("car_cmd_switch_node/cmd", car_cmd_topic)], ...)
```

The `remappings` line is a safety net for robots whose kinematics node reads another topic name; notebook 11 explains remapping. The launch file is also what the recipe's launcher runs when you use `dts code workbench`: `ros2 launch ros2_basics drive_node.launch.py veh:=ROBOTNAME`.

## Run the drive node

Two ways exist to run the node against your robot. Both need the robot's ROS 2 stack up (notebook 08).

**From the editor** (fastest while developing): in a terminal pointed at the robot's router (the `export ZENOH_CONFIG_OVERRIDE=...` line of notebook 08), after a build:

```bash
ros2 launch ros2_basics drive_node.launch.py veh:=ROBOTNAME
```

**From your laptop, on the robot** (how the exercise is meant to be deployed): in a terminal of your laptop, from the root of this repository:

```bash
dts code build -R ROBOTNAME
dts code workbench -R ROBOTNAME [-m]
```

with `-m` when the robot is virtual and the Duckiematrix is running. The workbench builds your `packages/` in a Docker image, sends it to the robot and runs the launcher of the exercise, which is the same `ros2 launch` command.

Either way, the terminal shows:

```shell
[INFO] [ROBOTNAME.drive_node]: robot 'ROBOTNAME': wheel baseline 0.1 m, radius 0.0318 m (defaults)
[INFO] [ROBOTNAME.drive_node]: ready: publishing on '/ROBOTNAME/car_cmd_switch_node/cmd' once started
```

Nothing moves yet: `enabled` is `false`. Check the node's interfaces from another terminal pointed at the robot (same `export`):

```bash
ros2 node info /ROBOTNAME/drive_node
ros2 param list /ROBOTNAME/drive_node
ros2 service list -t | grep drive_node
```

The services appear as:

```shell
/ROBOTNAME/drive_node/start [std_srvs/srv/Trigger]
/ROBOTNAME/drive_node/stop [std_srvs/srv/Trigger]
```

## Drive from the terminal

Put the robot on the floor with space in front of it (or watch the Duckiematrix window). Observe the commands as they are published, in one terminal:

```bash
ros2 topic echo /ROBOTNAME/car_cmd_switch_node/cmd
```

and, in another, start driving:

```bash
ros2 service call /ROBOTNAME/drive_node/start std_srvs/srv/Trigger
```

```shell
response:
std_srvs.srv.Trigger_Response(success=True, message='driving with v=0.2 m/s, omega=0.0 rad/s')
```

The robot drives straight for `max_duration` = 5 seconds and stops by itself; the echo shows ten commands per second with `v: 0.2`, then one with `v: 0.0`. Change the parameters and start again:

```bash
ros2 param set /ROBOTNAME/drive_node omega 2.0
ros2 param set /ROBOTNAME/drive_node v 0.1
ros2 service call /ROBOTNAME/drive_node/start std_srvs/srv/Trigger
```

The robot now turns left while advancing slowly. `ros2 service call /ROBOTNAME/drive_node/stop std_srvs/srv/Trigger` stops it at any time.

## Drive from the notebook

The same parameter and service interfaces are what the notebook panels use. Fill in your robot's name and run the cells; the panels are live and work as long as `drive_node` runs (in the editor terminal or in the workbench).

In [ ]:
ROBOT = "ROBOTNAME"   # <-- the name of your (real or virtual) Duckiebot

In [ ]:
from lx_ros2_tools.session import RosSession
from lx_ros2_tools.panel import ParamPanel, ActionButtons, ImageView, TopicPlot

ros = RosSession(ROBOT)

In [ ]:
panel = ParamPanel(ros, "drive_node", {"v": (-0.5, 0.5, 0.01), "omega": (-4.0, 4.0, 0.1), "max_duration": (0.0, 30.0, 0.5)})

`ParamPanel` reads the current values from the node (`get_parameters` service) and sends new ones with the *Send* button (`set_parameters` service): it is `ros2 param get/set` with sliders. Values are only applied when you press *Send*.

In [ ]:
buttons = ActionButtons(ros, "drive_node", ["start", "stop"])

`ActionButtons` calls the `Trigger` services `drive_node/start` and `drive_node/stop`; the response message of the node is shown under the buttons. Watch the robot (or its camera) and the encoders while it drives:

In [ ]:
camera = ImageView(ros, "camera_node/image/compressed")
ticks = TopicPlot(ros, "left_wheel_encoder_driver_node/tick", "duckietown_msgs/msg/WheelEncoderStamped", ["data"],
                  sensor=True, title="left wheel encoder ticks")

Set `v` to `0.2` and `omega` to `0`, press *Send*, then *start*: the tick count rises for 5 seconds. Set `omega` to `2.0` and start again: the robot drives in a circle (on the ground, a positive `omega` turns left). The same can be observed as `WheelsCmdStamped` messages produced by the kinematics node, which is what reaches the motors:

In [ ]:
wheels = TopicPlot(ros, "wheels_driver_node/wheels_cmd", "duckietown_msgs/msg/WheelsCmdStamped", ["vel_left", "vel_right"],
                   title="wheel commands produced by kinematics_node")

With `omega > 0` the right wheel turns faster than the left one, which is how a differential-drive robot turns left. If the plot stays empty while the commands are published, the kinematics node is not reading your topic: see the README's troubleshooting section.

In [ ]:
camera.close(); ticks.close(); wheels.close()

## Exercise: make the robot wiggle

The command published by `loop()` is constant. Modify the block between `YOUR CODE BELOW THIS LINE` and `YOUR CODE ABOVE THIS LINE` so that the robot alternates between turning left and turning right every second, while advancing at `v`: use `elapsed` (seconds since `start`) and the value of `self.params.omega` as the amplitude.

<details>
<summary>Hint</summary>

`int(elapsed) % 2` is `0` during the first second, `1` during the second one, `0` again during the third... A sign that depends on it is `1.0 if int(elapsed) % 2 == 0 else -1.0`.

</details>

Then:

1. save the file, rebuild (`colcon build --base-paths "$LX_DIR/packages" --packages-select ros2_basics` in `$COLCON_WS`), or `dts code build -R ROBOTNAME`;
2. stop the running `drive_node` with `Ctrl-C` and start it again (the running process still executes the old code);
3. set `omega` to `2.0` and `max_duration` to `6`, press *Send*, then *start*. The robot should snake forward.

The constructor established the node's interfaces and timer, while the executor invoked the callbacks that filled and published each message. Parameters and services let the terminal and the notebook control the running node, and a change in the source became visible after a rebuild and a restart.

## Further reading

The ROS 2 Jazzy tutorials on [writing a Python publisher and subscriber](https://docs.ros.org/en/jazzy/Tutorials/Beginner-Client-Libraries/Writing-A-Simple-Py-Publisher-And-Subscriber.html), [services](https://docs.ros.org/en/jazzy/Tutorials/Beginner-Client-Libraries/Writing-A-Simple-Py-Service-And-Client.html) and [parameters](https://docs.ros.org/en/jazzy/Tutorials/Beginner-Client-Libraries/Using-Parameters-In-A-Class-Python.html) provide more examples of nodes, timers, callbacks and spinning.

Let's move to [the next notebook about subscribers](../notebooks/10_subscribers_and_diagnosis.ipynb).